---
---
# **Procesamiento Automático de Balances Generales y Estados de Resultados Mediante un Modelo de Inteligencia Artificial**
---
# Prueba de Concepto sobre Google AI Documents

---
---
##Universidad Alfonso X El Sabio
##Master en Inteligencia Artificial 2025/2026
---
##Trabajo de Fin de Master (TFM)
---
**Autor:** Raúl Sánchez -
**eMail:** rsancgom@myuax.com -
**NP:** 867196

**Tutor:** Ángel Manuel Rayo Acevedo -
**eMail:** arayoace@uax.es

**Fecha de Entrega:** 22-Junio-2026

---

## Objetivo

Construir una Prueba de Concepto (POC, por sus siglas en inglés) con el uso de la tecnología Google AI Documents, para extraer información financiera de empresas colombianas proveniente de archivos no editables con formatos y estructuras desconocidas, para convertirla posteriormente en datos manipulables que se puedan validar y cargar a estructuras estándar previamente definidas.

----
## Justificación
Una de los principales procesos en empresas que comercializan información empresarial, consiste en recopilar información empresarial de todo tipo para su correspondiente tratamiento, depuración, limpieza y volcado sobre bases de datos puras, comunmente llamadas Bases de Datos Unificada de Empresas, con formatos y estructuras estándar previamente definidas; esta información constituye el activo principal de esta compañías y es finalmente el producto que comercializan con sus diferentes clientes.

El mayor de los inconvenientes que se tiene corresponde al cargue manual de información financiera, específicamente Balances Generales y Estados de Resultados de las diferentes empresas en Colombia, provenientes de fuentes públicas de difícil legibilidad, en archivos PDF con imágenes no editables y con formatos y estructuras desconocidas.

Esta manualidad ocasiona una alta carga operativa derivada de la necesidad de disponer de un número importante de personas realizando estas labores, con la posibilidad de cometer errores frecuentes y con altos tiempos procesamiento debido a la transcripción de cada uno de los datos de entrada y a la interpretación de estos para llevarlos a las estructuras estándar definidas en formatos predefinidos en Excel, en donde manualmente se hacen validaciones de consistencia, totalización de cifras, comparación con las fuentes y cargue final a la Base de Datos Unificada de Empresas.

Con base en lo anterior y con el fin de cubrir estas debilidades, se planteó la posibilidad de implementar una prueba de concepto que a través de un modelo de Inteligencia Artificial apoyado en la tecnología de Google AI Documents, permita extraer de manera automática la información en cuestión, convertirla en datos manipulables y cargarla en estructuras estándar previamente definidas.

Las tecnologías de Inteligencia Artificial disponibles en la actualidad tales como AI-OCR (Optical Character Recognition basado en IA) y LLM (Large Language Model), pueden resolver, sin lugar a duda, la problemática asociada a procesos manuales como el descrito anteriormente, con altos niveles de precisión y bajos costos.

----
##Metodología
Para cumplir con el objetivo serán desarrolladas las siguientes actividades:
1) Actividades previas.
2) Análisis y extracción del archivo fuente con Google AI Documents.
    - Definición y consumo de los servicios de Google AI Documents.
    - Formateo de la información y cargue en dataset local.
    - Exportación de la información formateada a archivo Excel.
3) Interpretación y Mapeo del Contenido con Microsoft Azure OpenAI.
    - Lectura y limpieza del documento extraído.
    - Lectura de la taxonomía definida
    - Definición y configuración del LLM Azure OpenAI para interpretar los estados financieros leídos con AI-OCR según la taxonomía.
    - Mapeo Inteligente para interpretar los estados financieros leídos con AI-OCR según la taxonomía usando LLM Azure OpenAI.
    - Generación del archivo excel final completamente interpretado y mapeado.

----


----
# **Actividades previas**

----

In [71]:
# Instalar e importar las librerías requeridas para la POC

# Instalar librerías de Google AI Documents
!pip install google-cloud-documentai

# Instalar librerías para el LLM Azure OpenAI
!pip install openai pandas openpyxl tiktoken

# Importar otras librerías de propósito general
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import re
import json
from typing import List, Tuple, Dict

# Importar librerías para el LLM Azure OpenAI
import openai
from collections import defaultdict
from openai import OpenAI
from openai import AzureOpenAI

# Importar librerías de Google
from google.colab import auth
from google.cloud import documentai_v1 as documentai

In [72]:
# Autenticar con Google Cloud services

# Desasignar la variable de ambiente GOOGLE_APPLICATION_CREDENTIALS si fue previamente configurada
# para asegurar la autenticación con Google desde Colab
if "GOOGLE_APPLICATION_CREDENTIALS" in os.environ:
    del os.environ["GOOGLE_APPLICATION_CREDENTIALS"]
    print("GOOGLE_APPLICATION_CREDENTIALS environment variable unset.")

# Atenticar con Google
auth.authenticate_user()
print('Authenticated with Google Cloud.')

Authenticated with Google Cloud.


In [73]:
# Conectar con la unidad de drive de Google y establecer directorio de trabajo

print("Montando el directorio de trabajo ...\n")
from google.colab import drive
drive.mount('/content/drive')

# Asignar variable global para el directorio de trabajo
directorio_datos = '/content/drive/MyDrive/Colab Notebooks/'

print("Directorio de trabajo montado.")

Montando el directorio de trabajo ...

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio de trabajo montado.


In [74]:
# Configurar los nombres de los archivos a utilizar

# Archivo de alta complejidad, mediana legibilidad
#FILE_PATH = os.path.join(directorio_datos, 'PRUEBA_12.pdf')
#output_excel_path = os.path.join(directorio_datos, 'PRUEBA_12_Google101.xlsx')
#output_file = os.path.join(directorio_datos, 'PRUEBA_12_LLM101.xlsx')

# Archivo de baja complejidad, buena legibilidad
#FILE_PATH = os.path.join(directorio_datos, 'PRUEBA_14.pdf')
#output_excel_path = os.path.join(directorio_datos, 'PRUEBA_14_Google102.xlsx')
#output_file = os.path.join(directorio_datos, 'PRUEBA_14_LLM102.xlsx')

# Archivo de baja complejidad, muy mala legibilidad
#FILE_PATH = os.path.join(directorio_datos, 'PRUEBA_32.pdf')
#output_excel_path = os.path.join(directorio_datos, 'PRUEBA_32_Google103.xlsx')
#output_file = os.path.join(directorio_datos, 'PRUEBA_32_LLM103.xlsx')

# Archivo de alta complejidad, mediana legibilidad
FILE_PATH = os.path.join(directorio_datos, 'PRUEBA_03.pdf')
output_excel_path = os.path.join(directorio_datos, 'PRUEBA_03_Google104.xlsx')
output_file = os.path.join(directorio_datos, 'PRUEBA_03_LLM004.xlsx')

file_input = output_excel_path
file_taxonomy = os.path.join(directorio_datos, '03_Taxonomia-Base-BGyEERR.xlsx')

In [75]:
# Configurar los parámetros requeridos para el servicio Google AI Documents
# Esto requiere una configuración previa en Google Cloud Platform (GCP), así:
#   0. Asignar créditos en GPC a la cuenta con la que se va a hacer la POC (@gmail.com)
#   1. Crear un Proyecto: En la consola de Google Cloud, crear un proyecto nuevo.
#   2. Habilitar la API: En la barra de búsqueda de GCP, buscar Cloud Document AI API y hacer click en Habilitar.
#   3. Crear el Procesador (Processor):
#      * Entrar al módulo de Document AI.
#      * Seleccionar "Crear procesador (Document-OCR-RS)".
#      * Para documentos financieros generales o tablas, elige el procesador Form Parser (Analizador de formularios) o Document OCR
#      * Copiar el ID del procesador, la Región (ej. us o eu) y ID de Proyecto.
#  4. Autenticar con IAM (para POCs este paso puede reemplazarse por la uatenticación desde Colab con un usuario @gmail.com)
#     Si es para un proyecto empresarial, debe crearse una Organización en GCP y cuenta de facturación Google.
PROJECT_ID = "project-a75a8ec8-0bdf-4633-b9b"
LOCATION = "us"
PROCESSOR_ID = "c396c1ce23dc10a9"

# configuración Azure OpenAI
# Esto requiere una configuración previa en Azure Cloud
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://rs002.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2025-01-01-preview"
os.environ["AZURE_OPENAI_API_KEY"] = "7gfqfiCbfUbAfCwbkzcM13mK43TsHqxoFDD1f32YBYtqv34OvjrhJQQJ99CEACHYHv6XJ3w3AAABACOGxmO9"
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-02-15-preview"
DEPLOYMENT_NAME = "gpt-4o"

----
# **Análisis y Extracción del Archivo Fuente con Google AI Documents**

----

----
#Definición y consumo de los servicios de Google AI Document

----

In [76]:
# Construir la función para consumo del servicio Google AI Documents que permite análisis y extracción de información del archivo
def process_document(project_id, location, processor_id, file_path):
    # Inicializar el cliente de Document AI apuntando a la región correcta
    client_options = {"api_endpoint": f"{location}-documentai.googleapis.com"}
    client = documentai.DocumentProcessorServiceClient(client_options=client_options)

    # Obtener el nombre completo del recurso del procesador
    name = client.processor_path(project_id, location, processor_id)

    # Leer el archivo PDF local en formato binario
    with open(file_path, "rb") as image:
        image_content = image.read()

    # Configurar la solicitud (Request)
    raw_document = documentai.RawDocument(content=image_content, mime_type="application/pdf")
    request = documentai.ProcessRequest(name=name, raw_document=raw_document)

    # Envíar la solicitud a la API de Google Cloud
    result = client.process_document(request=request)
    return result.document


In [77]:
# Ejecutar el análisis y extracción de información del archivo fuente PDF
document_data = process_document(PROJECT_ID, LOCATION, PROCESSOR_ID, FILE_PATH)

----
#Formateo de la información y cargue en dataset local
----


In [78]:
# Construir función para transformar la salida a un dataframe de pandas con la siguiente estructura
#   pagina: número de página del documento
#	  fila: número que identifica la fila en donde debe ir el texto encontrado (para un archivo excel)
#   columna: letra que identifica la columna en donde debe ir el texto encontrado (para un archivo excel)
#   valor: texto encontrado y convertido en editable
#   Coordenadas del polígono en donde se encontró el texto:
#     x1,y1	(coordenada izquierda-inferior)
#     x2,y2	(coordenada derecha-inferior)
#     x3,y3	(coordenada derecha-superior)
#     x4,y4 (coordenada izquierda-superior)
def _text_from_anchor(doc_text: str, text_anchor) -> str:
    if text_anchor is None or not getattr(text_anchor, "text_segments", None):
        return ""
    parts = []
    for seg in text_anchor.text_segments:
        start = int(getattr(seg, "start_index", 0) or 0)
        end = int(getattr(seg, "end_index", 0) or 0)
        if end > start:
            parts.append(doc_text[start:end])
    return "".join(parts).strip()

def _get_vertices_normalized(bounding_poly, page_w, page_h):
    if bounding_poly is None:
        return []

    if getattr(bounding_poly, "normalized_vertices", None):
        return [(float(v.x), float(v.y)) for v in bounding_poly.normalized_vertices]

    if getattr(bounding_poly, "vertices", None):
        page_w = page_w or 1.0
        page_h = page_h or 1.0
        return [(float(v.x) / page_w, float(v.y) / page_h) for v in bounding_poly.vertices]

    return []

def _centroid_xy(verts):
    if not verts:
        return (0.0, 0.0)
    xs = [x for x, _ in verts]
    ys = [y for _, y in verts]
    return (sum(xs) / len(xs), sum(ys) / len(ys))

def _cluster_1d(values, tol):
    if not values:
        return {}

    sorted_vals = sorted(values)
    clusters = []

    for v in sorted_vals:
        if not clusters:
            clusters.append({"center": v, "members": [v]})
            continue

        if abs(v - clusters[-1]["center"]) <= tol:
            clusters[-1]["members"].append(v)
            clusters[-1]["center"] = sum(clusters[-1]["members"]) / len(clusters[-1]["members"])
        else:
            clusters.append({"center": v, "members": [v]})

    mapping = {}
    for v in values:
        best_i = min(range(len(clusters)), key=lambda i: abs(v - clusters[i]["center"]))
        mapping[v] = best_i + 1

    return mapping

def _num_to_col_letter(n):
    s = ""
    while n > 0:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s if s else "A"

def _extract_8_coords(verts):
    verts = verts[:4]
    while len(verts) < 4:
        verts.append((None, None))

    return {
        "x1": verts[0][0], "y1": verts[0][1],
        "x2": verts[1][0], "y2": verts[1][1],
        "x3": verts[2][0], "y3": verts[2][1],
        "x4": verts[3][0], "y4": verts[3][1],
    }

def _get_elements(page, level):
    if level == "line":
        if hasattr(page, "lines") and page.lines:
            return page.lines

    return getattr(page, "tokens", []) or []

def documentai_to_dataframe(document_data, x_tol=0.03, y_tol=0.015, level="token"):
    doc_text = getattr(document_data, "text", "") or ""
    rows = []

    for p_idx, page in enumerate(getattr(document_data, "pages", []) or [], start=1):
        dim = getattr(page, "dimension", None)
        page_w = float(getattr(dim, "width", 1.0) or 1.0) if dim else 1.0
        page_h = float(getattr(dim, "height", 1.0) or 1.0) if dim else 1.0

        elements = _get_elements(page, level)

        xs, ys = [], []
        tmp = []

        for el in elements:
            layout = getattr(el, "layout", None)
            if layout is None:
                continue

            text = _text_from_anchor(doc_text, layout.text_anchor)
            if not text:
                continue

            verts = _get_vertices_normalized(layout.bounding_poly, page_w, page_h)
            cx, cy = _centroid_xy(verts)

            xs.append(cx)
            ys.append(cy)
            tmp.append((text, verts, cx, cy))

        y_map = _cluster_1d(ys, y_tol)
        x_map = _cluster_1d(xs, x_tol)

        for text, verts, cx, cy in tmp:
            fila = y_map.get(cx, 1)
            fila = y_map.get(cy, 1)
            col_num = x_map.get(cx, 1)
            columna = _num_to_col_letter(col_num)

            coords = _extract_8_coords(verts)

            rows.append({
                "pagina": p_idx,
                "fila": int(fila),
                "columna": columna,
                "valor": text,
                **coords
            })

    df = pd.DataFrame(rows, columns=[
        "pagina", "fila", "columna", "valor",
        "x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4"
    ])

    if not df.empty:
        df = df.sort_values(by=["pagina", "fila", "columna"]).reset_index(drop=True)

    return df


In [79]:
# Transformar la salida a un dataframe de pandas

df = documentai_to_dataframe(document_data, x_tol=0.03, y_tol=0.015, level="line")

df.head(10)

,pagina,fila,columna,valor,x1,y1,x2,y2,x3,y3,x4,y4
0,1,1,D,MUNICIPIO DE LA ESTRELLA,0.347554,0.068571,0.583618,0.068571,0.583618,0.078681,0.347554,0.078681
1,1,2,D,ESTADO DE SITUACION FINANCIERA,0.313993,0.084396,0.613766,0.084396,0.613766,0.095385,0.313993,0.095385
2,1,3,D,SEPTIEMBRE 30 DE 2023,0.362912,0.100659,0.563709,0.100659,0.563709,0.111209,0.362912,0.111209
3,1,4,D,(Cifras en pesos),0.391923,0.116484,0.529579,0.116484,0.529579,0.130549,0.391923,0.130549
4,1,5,E,jun-23,0.521615,0.147253,0.572810,0.147253,0.572810,0.159121,0.521615,0.159121
5,1,5,G,sep-23,0.715017,0.147692,0.766780,0.147692,0.766780,0.159121,0.715017,0.159121
6,1,6,B,Código ACTIVO CORRIENTE,0.096701,0.164835,0.311149,0.164835,0.311149,0.176703,0.096701,0.176703
7,1,6,E,252.163.930.820,0.526735,0.166154,0.643914,0.166154,0.643914,0.175385,0.526735,0.175385
8,1,6,G,280.380.428.180,0.717292,0.166593,0.836746,0.166593,0.836746,0.175824,0.717292,0.175824
9,1,7,C,11 EQUIVALENTE AL EFECTIVO,0.135381,0.199560,0.370876,0.199560,0.370876,0.209231,0.135381,0.209231


----
#Exportación de la información formateada a archivo Excel
----


In [80]:
# Construir función para convertir el dataframe a un archivo excel con la estructura del PDF leído

def dataframe_to_excel(df: pd.DataFrame, output_file: str = "output.xlsx"):

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        # Hoja 1: DataFrame completo
        df.to_excel(writer, sheet_name="dataframe", index=False)

        # Obtener páginas únicas
        paginas = sorted(df["pagina"].dropna().unique())

        # Crear una hoja por página
        for pagina in paginas:
            df_page = df[df["pagina"] == pagina]

            # Crear hoja vacía tipo matriz
            sheet_name = f"pagina_{pagina}"
            worksheet = writer.book.create_sheet(title=sheet_name)
            writer.sheets[sheet_name] = worksheet

            for _, row in df_page.iterrows():
                fila_excel = int(row["fila"])
                columna_excel = row["columna"]

                # Escribir el valor en celda tipo Excel
                worksheet[f"{columna_excel}{fila_excel}"] = row["valor"]

    print(f"Archivo Excel generado: {output_file}")

In [81]:
# Convertir el dataframe a un archivo excel con la estructura del PDF leído

dataframe_to_excel(df, output_excel_path)

print(f"¡Procesamiento completado con éxito! El archivo editable se ha guardado en: {output_excel_path}")

Archivo Excel generado: /content/drive/MyDrive/Colab Notebooks/PRUEBA_03_Google104.xlsx
¡Procesamiento completado con éxito! El archivo editable se ha guardado en: /content/drive/MyDrive/Colab Notebooks/PRUEBA_03_Google104.xlsx


---
# **Interpretación y Mapeo del Contenido con Microsoft Azure OpenAI**

---

---
#Lectura y limpieza del documento extraído

---

In [82]:
# Leer y limpiar desde Excel (múltiples hojas)
# NOTA: SE INTENTÓ USAR OPENAI PARA LEER DIRECTAMENTE EL PDF PERO EL PROCESO REQUIERE PREVIAMENTE PROCESARLO CON AI-OCR

# Función para crear texto plano a partir del archivo Excel generado por el proceso de AI-OCR
def extract_text_from_excel(file_path):
    text = ""
    xls = pd.ExcelFile(file_path)

    for sheet_name in xls.sheet_names:
        # Excluir hojas que no se requieren para la interpretación
        if sheet_name in ["dataframe"]:
            continue

        df = pd.read_excel(file_path, sheet_name=sheet_name, dtype=str)
        df = df.fillna("")

        for _, row in df.iterrows():
            row_text = " ".join([str(cell) for cell in row if str(cell).strip() != ""])
            if row_text:
                text += row_text + "\n"

    # Limpieza básica
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'Página \d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s{2,}', ' ', text)

    return text

# Ejecutar la función para crear texto plano a partir del archivo Excel generado por el proceso de AI-OCR
excel_text = extract_text_from_excel(file_input)

----
#Lectura de la taxonomía definida

----

In [83]:
# Leer la taxonomía resultado definida para el balance general
bg_taxonomy = pd.read_excel(file_taxonomy,
                           sheet_name="Taxonomia-Base-BG")

# Leer la taxonomía resultado definida para el estado de resultados
eerr_taxonomy = pd.read_excel(file_taxonomy,
                             sheet_name="Taxonomia-Base-EERR")

# Tomar los nombres de las cuentas de las taxonomías
bg_accounts = bg_taxonomy.iloc[:,0].dropna().tolist()
eerr_accounts = eerr_taxonomy.iloc[:,0].dropna().tolist()

----
# Definición y configuración del LLM Azure OpenAI para interpretar los estados financieros leídos con AI-OCR según la taxonomía

----

In [84]:
# Definir y configurar el LLM Azure OpenAI para interpretar los estados financieros leídos con AI-OCR según la taxonomía

# Asignar los parámetros requeridos por Azure OpenAI
client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)

# Función para usar Azure OpenAI (mapeo inteligente)
def map_accounts_with_llm(text, taxonomy, tipo_estado):
    prompt = f"""
Eres un experto contable colombiano.

Tienes un texto extraído de un estado financiero desordenado.
Debes mapear cada cuenta a una taxonomía estándar.

TIPO: {tipo_estado}

TAXONOMÍA:
{taxonomy}

REGLAS:
- Identifica cuentas y valores numéricos por cada periodo
- Trata de ajustar las cuentas de acuerdo con el nombre en la taxonomía que es de la forma "GRUPO - SUBGRUPO - CUENTA"
- Asume como válida la cuenta cuyo nombre esté completa o parcialmente en la taxonomía
- Asume como válida la cuenta cuyo nombre contenga completa o parcialmente el nombre en la taxonomía
- No tengas en cuenta las diferencias por mayúsculas y minúsculas
- Respeta el agrupamiento de las cuentas de acuerdo como se indica en la taxonomía
- El nombre de la cuenta que se genere en el resultado debe tener el nombre de la cuenta original y el nomnbre de la cuenta en la taxonomía
- Ignora encabezados, logos, ruido
- Ten en cuenta todas y cada una de las cuentas; Si no encuentras coincidencia, asigna:
  "SIN IDENTIFICAR (texto original con periodos y valores)"
- Mantén valores numéricos
- Responde SOLO en JSON así:

[
  {{
    "cuenta_original": "...",
    "cuenta_taxonomia": "...",
    "periodo": ...,
    "valor": ...
  }}
]

TEXTO:
{text[:12000]}
"""

    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": "Eres un analista financiero experto en NIIF Colombia"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    raw_content = response.choices[0].message.content
    start_idx = raw_content.find('[')
    end_idx = raw_content.rfind(']')

    if start_idx != -1 and end_idx != -1 and start_idx < end_idx:
        json_string = raw_content[start_idx : end_idx + 1]
    else:
        print(f"Alerta: los delimitadores del JSON '[' and ']' no se encontraron o se perdieron en la respuesta del LLM. Intentando un string simple. Content:\n{raw_content}")
        json_string = raw_content.strip('```json').strip('```').strip()

    return json_string.strip()

----
# Mapeo Inteligente para interpretar los estados financieros leídos con AI-OCR según la taxonomía usando LLM Azure OpenAI

----

In [85]:
# Ejecutar función para usar Azure OpenAI (mapeo inteligente)
bg_json = map_accounts_with_llm(excel_text, bg_accounts, "BALANCE GENERAL")
eerr_json = map_accounts_with_llm(excel_text, eerr_accounts, "ESTADO DE RESULTADO")

# Leer los JSON resultado del mapeo inteligente
bg_data = json.loads(bg_json)
eerr_data = json.loads(eerr_json)

In [86]:
# Mostrar resultado del mapeo inteligente del balance general
print (bg_json)
print(bg_data)

[
  {
    "cuenta_original": "ACTIVO CORRIENTE",
    "cuenta_taxonomia": "SIN IDENTIFICAR (ACTIVO CORRIENTE)",
    "periodo": "jun-23",
    "valor": 252163930820
  },
  {
    "cuenta_original": "ACTIVO CORRIENTE",
    "cuenta_taxonomia": "SIN IDENTIFICAR (ACTIVO CORRIENTE)",
    "periodo": "sep-23",
    "valor": 280380428180
  },
  {
    "cuenta_original": "EQUIVALENTE AL EFECTIVO",
    "cuenta_taxonomia": "SIN IDENTIFICAR (EQUIVALENTE AL EFECTIVO)",
    "periodo": "jun-23",
    "valor": 97793793418
  },
  {
    "cuenta_original": "EQUIVALENTE AL EFECTIVO",
    "cuenta_taxonomia": "SIN IDENTIFICAR (EQUIVALENTE AL EFECTIVO)",
    "periodo": "sep-23",
    "valor": 90591921820
  },
  {
    "cuenta_original": "Caja",
    "cuenta_taxonomia": "ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA, BANCOS)",
    "periodo": "jun-23",
    "valor": 97784293418
  },
  {
    "cuenta_original": "Caja",
    "cuenta_taxonomia": "ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA, BANCOS)",
    "periodo": "sep-23"

In [87]:
# Mostrar resultado del mapeo inteligente del estado de resultado
print(eerr_json)
print(eerr_data)

[
  {
    "cuenta_original": "4 INGRESOS Concepto",
    "cuenta_taxonomia": "INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVICIOS)",
    "periodo": "sep-23",
    "valor": 169688181327
  },
  {
    "cuenta_original": "41 Ingresos Fiscales",
    "cuenta_taxonomia": "INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVICIOS)",
    "periodo": "sep-23",
    "valor": 112841616989
  },
  {
    "cuenta_original": "410507 Impuesto predial unificado",
    "cuenta_taxonomia": "INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVICIOS)",
    "periodo": "sep-23",
    "valor": 15326233093.35
  },
  {
    "cuenta_original": "410519 Impuesto de delineacion urbana",
    "cuenta_taxonomia": "INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVICIOS)",
    "periodo": "sep-23",
    "valor": 1673130739.2
  },
  {
    "cuenta_original": "410535 Sobretasa a la gasolina",
    "cuenta_taxonomia": "INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVICIOS)",
    "periodo": "sep-23",
    "valor": 2753053000
  },
  {
    "cuenta_original": "410545

In [88]:
# Normalizar resultados (incluye SIN IDENTIFICAR)

# Función para normalizar los resultados
def normalize_results(data):
    rows = []

    for item in data:
        cuenta = item.get("cuenta_taxonomia", "")
        valor = item.get("valor", 0)
        periodo = item.get("periodo", "")
        original = item.get("cuenta_original", "")

        if "SIN IDENTIFICAR" in cuenta:
            #cuenta = f"SIN IDENTIFICAR ({original})"
            cuenta = "SIN IDENTIFICAR"

        rows.append({
            "Cuenta": cuenta,
            "Original": original,
            "Periodo": periodo,
            "Valor": valor
        })

    return pd.DataFrame(rows)

# Ejecutar la función para normalizar los resultados del balance general y del estado de resultados
df_bg = normalize_results(bg_data)
df_eerr = normalize_results(eerr_data)

In [89]:
# Mostrar el dataset con la normalización del balance general
df_bg

,Cuenta,Original,Periodo,Valor
0,SIN IDENTIFICAR,ACTIVO CORRIENTE,jun-23,2.521639e+11
1,SIN IDENTIFICAR,ACTIVO CORRIENTE,sep-23,2.803804e+11
2,SIN IDENTIFICAR,EQUIVALENTE AL EFECTIVO,jun-23,9.779379e+10
3,SIN IDENTIFICAR,EQUIVALENTE AL EFECTIVO,sep-23,9.059192e+10
4,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",Caja,jun-23,9.778429e+10
5,"ACTIVOS - ACTIVOS CORRIENTES - EFECTIVO (CAJA,...",Caja,sep-23,9.058242e+10
6,SIN IDENTIFICAR,Efectivo de uso restringido,jun-23,0.000000e+00
7,SIN IDENTIFICAR,Efectivo de uso restringido,sep-23,0.000000e+00
8,ACTIVOS - ACTIVOS CORRIENTES - CUENTAS POR COBRAR,CUENTAS POR COBRAR,jun-23,7.072205e+10
9,ACTIVOS - ACTIVOS CORRIENTES - CUENTAS POR COBRAR,CUENTAS POR COBRAR,sep-23,8.162574e+10


In [90]:
# Mostrar el dataset con la normalización del estado de resultados
df_eerr

,Cuenta,Original,Periodo,Valor
0,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,4 INGRESOS Concepto,sep-23,1.696882e+11
1,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,41 Ingresos Fiscales,sep-23,1.128416e+11
2,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,410507 Impuesto predial unificado,sep-23,1.532623e+10
3,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,410519 Impuesto de delineacion urbana,sep-23,1.673131e+09
4,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,410535 Sobretasa a la gasolina,sep-23,2.753053e+09
5,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,410545 Impuesto alumbrado publico,sep-23,3.039160e+09
6,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,410558 Publicidad exterior visual,sep-23,4.351122e+06
7,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,410562 Sobretasa bomberil,sep-23,2.142348e+09
8,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,4110 410576 Estampillas No tributarios,sep-23,1.919940e+10
9,INGRESOS - INGRESOS (VENTAS DE BIENES Y SERVIC...,44 Transferencias Devoluciones y descuentos,sep-23,5.684656e+10


----
# Generación del archivo excel final completamente interpretado y mapeado

----

In [91]:
# Generar Excel Final

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_bg.to_excel(writer, sheet_name="BG", index=False)
    df_eerr.to_excel(writer, sheet_name="EERR", index=False)

print("Archivo generado:", output_file)

Archivo generado: /content/drive/MyDrive/Colab Notebooks/PRUEBA_03_LLM004.xlsx
